# Demo 01 - Summary Analysis

In addition to our work reviewing the data model, we can perform a summary analysis of our data in Python. At this level, we are primarily interested in counts, sums, mins, maxes, ranges, and the like rather than breakouts by variable.

To do this, we will rely heavily on Python packages, which we can install using <code>pip</code>.

## Data Retrieval

First, we will need to retrieve our data.  We can use the `pyodbc` package to do this.  We will subsequently run summary statistics on each key data set.

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import tomllib

with open("../config.toml", "rb") as f:
    config = tomllib.load(f)
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
engine = create_engine(f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes")


In [ ]:
buses = pd.read_sql("""SELECT
    b.BusID,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    b.AverageRoadConditions,
    b.DateFirstInService,
    b.DateRetired
FROM dbo.Bus b
    INNER JOIN dbo.BusModel bm
        ON bm.BusModelID = b.BusModelID;""", engine)
employees = pd.read_sql("SELECT * FROM dbo.Employee", engine)
expense_categories = pd.read_sql("SELECT * FROM dbo.ExpenseCategory", engine)
vendors = pd.read_sql("SELECT * FROM dbo.Vendor", engine)
vendor_expense_categories = pd.read_sql("""SELECT
	vec.VendorID,
	vec.ExpenseCategoryID,
	v.VendorName,
	ec.ExpenseCategory
FROM dbo.VendorExpenseCategory vec
	INNER JOIN dbo.Vendor v
		ON vec.VendorID = v.VendorID
	INNER JOIN dbo.ExpenseCategory ec
		ON vec.ExpenseCategoryID = ec.ExpenseCategoryID;""", engine)
line_items = pd.read_sql("""SELECT
    li.LineItemID,
    li.LineItemDate,
    li.BusID,
    li.ExpenseCategoryID,
    li.VendorID,
    li.EmployeeID,
    li.CountersignerEmployeeID,
    li.Amount
FROM dbo.LineItem li
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID;""", engine)

Now that we have our data, let's take a look at it.

## Buses

In [ ]:
buses[0:5]

In [ ]:
buses.describe(include='all')

Things don't quite look right with respect to the dates, so let's see why.  The `dtypes` property gives us a breakdown of each data type.

In [ ]:
buses.dtypes

It turns out that `DateFirstInService` and `DateRetired` come back as strings (represented as `object` here) rather than proper dates.  So let's fix that.

In [ ]:
buses['DateFirstInService'] = pd.to_datetime(buses['DateFirstInService'])
buses['DateRetired'] = pd.to_datetime(buses['DateRetired'])

In [ ]:
buses.describe(include='all')

Now things are looking better!

Key takeaways:

1. We have 1560 buses in our data set.
2. Average Road Conditions ranges from 0.5 to 0.92 and represents how bad the roads are, such as potholes, road material, etc.
3. Of the 1560 buses which have ever been in service, 490 have been retired, which means that there are currently 1070 buses in active duty.
4. There are five models of bus in service.  Each bus model has its own quality, ranging from 0.71 to 0.91 and represents a quantitative measure of how likely the bus is to break down.  Lower numbers are worse.

## Employees

In [ ]:
employees

We have 12 employees in total.

## Expense Categories

In [ ]:
expense_categories.sort_values('ExpenseCategoryID')

We have 28 expense categories.  Each one has its own rough price, but we don't see any of that information directly in the database, as there are different vendors who offer up different prices depending on market circumstances.

## Vendors

In [ ]:
vendors.sort_values('VendorID')

We have 15 vendors in total.  As we can surmise from the names, these vendors have their own specialties.  We can see that in:

## Vendor Expense Categories

In [ ]:
vendor_expense_categories

This is a listing of which venders offer which services.

We can see how many different categories each vendor offers:

In [ ]:
vendor_expense_categories[["VendorName", "ExpenseCategory"]] \
    .groupby(by='VendorName') \
    .count() \
    .sort_values('ExpenseCategory', ascending = False)

And we can see the converse:  sole-source suppliers.  In an audit, we might investigate the reason why we would have sole-source suppliers in these categories.  It's not necessarily a bad thing--after all, there may not be any competition for certain niche parts or the agency might have negotiated with a particular seller to get a superior discount.

In [ ]:
vendor_expense_categories[["VendorName", "ExpenseCategory"]] \
    .groupby(by='ExpenseCategory') \
    .count() \
    .sort_values('VendorName')

Based on this, we can see that there are sole-source suppliers for four categories.

## Line Items

In [ ]:
line_items['LineItemDate'] = pd.to_datetime(line_items['LineItemDate'])
line_items.describe(include='all')

Looking at our line item fact, we have a few findings here:

1. The mean line item amount is for 195.97 USD and the median 109.62 USD.  This means that we skew right, with some particularly expensive items driving up the mean price.
2. The maximum amount is 3340.68 USD, but the 75th percentile is only 225.57 USD.
3. Of the 241,228 invoices, only 5389 have a countersigner employee ID.  We know that only line items totaling over 1000 USD require a countersigner, so this gives us an idea of how infrequently we hit that mark:  about 2% of the time.  It might be worth considering whether this is a valuable control or if the minimum amount should go down a bit.

## Conclusions

At this point, we are starting to become familiar with the data set.  The data structure looks fairly close to a fact (line items) and a series of dimensions (bus, vendor, employee, expense category).

From here, we want to carry on with further analyses, including **Growth Analysis**.